# Debugging EEG Pipeline: Mengapa Data Menjadi None?
Notebook ini meniru **persis** alur (pipeline) aplikasi web untuk melacak di mana data subjek terbuang atau tidak memenuhi syarat (diekstrak kosong/None).

In [14]:
# 1. Import Modul dan Requirements yang persis sama dengan Web App
import pandas as pd
import numpy as np
import warnings

# Mute warnings agar terminal rapi
warnings.filterwarnings('ignore')

import sys
import os
sys.path.append(os.path.abspath('.'))

from processing.loader import EEGLoader
from processing.features import EEGFeatures
from config import DEFAULT_SUBBANDS, DEFAULT_FEATURES

print("Modul berhasil di-load!")

Modul berhasil di-load!


## 1. Definisikan File Uji Coba
Ganti string di bawah ini dengan path absolut ke file EDF (misal: ID3/ALS3) yang bermasalah (hasilnya None).

In [15]:
# Ganti dengan path ke file target Anda 
# Misal: "C:/Users/.../dummy/ALS03/time1/scenario1/H.edf"
TARGET_EDF_PATH = r"C:\Users\Henry\Documents\KULIAH\riset\EEG\dummy.zip" # <--- UBAH INI

# Inisialisasi Loader dan baca file
loader = EEGLoader()
print(f"Loading data: {TARGET_EDF_PATH}")

try:
    info = loader.load_edf(TARGET_EDF_PATH)
    df = loader.extract_dataframe()
    print("Channel tersedia:", loader.channel_names)
    print("Sampling Frequency (sfreq):", loader.sfreq, "Hz")
    print("Total panjang rekaman:", len(df), "sampel (", len(df)/loader.sfreq, "detik)")
except Exception as e:
    print("Gagal meload file EDF:", e)

Loading data: C:\Users\Henry\Documents\KULIAH\riset\EEG\dummy.zip
Gagal meload file EDF: Gagal memuat file EDF: Only EDF files are supported, got zip.


## 2. Pengecekan Durasi Tiap Occurrence / Task
Penyumbang error utama biasanya adalah **Occurrence 1** yang durasinya terpotong drastis (kurang dari 1 detik atau bahkan berisi 4 titik sampel). Mari kita periksa.

In [16]:
# Dapatkan seluruh ringkasan Task (Marker) yang ada di titik EDF 
occurrences = loader.get_task_occurrences()

if not occurrences:
    print("Tidak ditemukan Task/Marker! Ekstraksi pasti gagal/kosong.")
else:
    print("=== DAFTAR TASK DAN DURASI OCCURRENCE ===")
    for occ in occurrences:
        task_name = occ["task"]
        occ_num = occ["occurrence"]
        
        # Ekstrak rentang potongan data task/occurrence tersebut menggunakan engine app
        seg = loader.extract_occurrence_segment(df, task_name, occ_num)
        
        # Hitung panjangnya
        num_samples = len(seg)
        durations_sec = num_samples / loader.sfreq
        
        # Cek status syarat algoritma di web (syarat: minimal ada bbrp point sampel, app limit: 4 sampel atau 0.5 sec)
        threshold = min(100, loader.sfreq * 0.5)
        status = "✅ LULUS (Valid)" if num_samples >= threshold else "❌ GAGAL (Terlalu Pendek / Kosong)"
        
        print(f"Task: {task_name:10} | Occ: {occ_num} | Sampel: {num_samples:5} | Durasi: {durations_sec:.3f} s -> {status}")

Tidak ditemukan Task/Marker! Ekstraksi pasti gagal/kosong.


## 3. Simulasi Ekstraksi Fitur (Step by Step Murni)
Aplikasi memanggil metode di bawah ini untuk mengambil perwakilan nilai bagi setiap task (contoh "Resting"). Mari kita simulasikan untuk melihat apakah hasilnya kosong atau sukses di EEGFeatures.

In [17]:
channels_to_extract = ["C3", "C4", "Cz"] # Gunakan beberapa channel yg diuji

# Ambil list task unik dari file target
tasks = list(set([occ["task"] for occ in occurrences])) if occurrences else []

# Simulasi seperti function: compute_first_occurrence_features()
for task_name in tasks:
    print(f"\n---> Mengekstrak Task: {task_name}")
    
    # Ambil list nomor occurrence untuk task ini
    occ_nums = [occ["occurrence"] for occ in occurrences if occ["task"] == task_name]
    
    success = False
    for occ_num in occ_nums:
        seg = loader.extract_occurrence_segment(df, task_name, occ_num)
        threshold = min(100, loader.sfreq * 0.5)
        
        if seg.empty or len(seg) < threshold:
            print(f"   [Skip] Occ {occ_num} durasinya tidak memenuhi syarat ({len(seg)} sampel)")
            continue
            
        print(f"   [Proses] Menggunakan Occ {occ_num} ({len(seg)} sampel). Mencoba hitung fitur...")
        
        # Proses perhitungannya persis seperti backend di web
        feat_df = EEGFeatures.compute_subband_features(
            seg, channels_to_extract, loader.sfreq, DEFAULT_SUBBANDS, DEFAULT_FEATURES,
            include_frequency=True
        )
        
        if not feat_df.empty:
            print(f"   [SUKSES!] Fitur didapatkan untuk {task_name} pada Occ {occ_num}!")
            display(feat_df.head())
            success = True
            break
        else:
            print(f"   [GAGAL] Perhitungan numpy/MNE mereturn value kosong/None!")
            
    if not success:
        print(f"   [FATAL] Tidak ada satu pun occurrence dari task '{task_name}' yang valid. Ini penyebab datanya berstatus None pada Excel!")

## 4. Diagnostik Batch: Scan SEMUA File EDF dari ZIP
Cell ini akan memproses **seluruh file EDF** dari ZIP (persis seperti web) dan melaporkan mana yang sukses, mana yang gagal, dan **mengapa** gagal.

In [18]:
import zipfile, io
from processing.filters import EEGFilters
from config import DEFAULT_SUBBANDS, DEFAULT_FEATURES

# ====== GANTI PATH ZIP ANDA DI SINI ======
ZIP_PATH = r"C:\Users\Henry\Documents\KULIAH\riset\EEG\dummy.zip"  # <--- UBAH INI

# Channel yang dipakai di web
CHANNELS = ["C3", "C4", "Cz"]

# Subband yang dipilih user (High_Beta, Low_Beta, Mu)
SELECTED_SUBBANDS = {
    "Mu": (8, 12),
    "Low_Beta": (12, 16),
    "High_Beta": (20, 30),
}

# Filter settings (sesuai pilihan user di web)
USE_AMPLITUDE = True   # Amplitude filter ON
USE_ICA = True          # ICA ON

with open(ZIP_PATH, "rb") as f:
    zip_bytes = f.read()

edf_list = EEGLoader.list_edf_in_zip(io.BytesIO(zip_bytes))
print(f"Total file EDF ditemukan: {len(edf_list)}\n")

results = []

for edf_path in edf_list:
    meta = EEGLoader.detect_category(edf_path)
    
    # Filter hanya time1 (sesuai pilihan user)
    if meta["time"] != "time1":
        continue
    
    loader = EEGLoader()
    
    # 1. Load file
    try:
        buf = io.BytesIO(zip_bytes)
        loader.load_edf_from_zip(buf, edf_path)
    except Exception as e:
        results.append({"file": edf_path, "subject": meta["subject"], "status": "GAGAL LOAD", "reason": str(e)})
        continue
    
    # 2. Channel check
    ch_list = [c for c in CHANNELS if c in loader.channel_names]
    if not ch_list:
        results.append({"file": edf_path, "subject": meta["subject"], "status": "GAGAL", "reason": f"Channel {CHANNELS} tidak ada. Tersedia: {loader.channel_names[:5]}"})
        loader._cleanup_tmp()
        continue
    
    # 3. Amplitude Filter (sama seperti web)
    if USE_AMPLITUDE:
        try:
            EEGFilters.apply_amplitude_filter(loader)
        except Exception as e:
            print(f"  [WARN] Amplitude filter gagal untuk {edf_path}: {e}")
    
    # 4. Bandpass
    try:
        low_all = min(v[0] for v in SELECTED_SUBBANDS.values())
        high_all = max(v[1] for v in SELECTED_SUBBANDS.values())
        EEGFilters.apply_bandpass(loader, low_all, high_all)
    except Exception as e:
        results.append({"file": edf_path, "subject": meta["subject"], "status": "GAGAL FILTER", "reason": str(e)})
        loader._cleanup_tmp()
        continue
    
    # 5. ICA (sama seperti web)
    if USE_ICA:
        try:
            EEGFilters.apply_ica(loader)
        except Exception as e:
            print(f"  [WARN] ICA gagal untuk {edf_path}: {e}")
    
    # 6. Extract DataFrame
    df = loader.extract_dataframe()
    tasks_found = [t for t in loader.get_task_list() if t != "none"]
    
    if not tasks_found:
        results.append({"file": edf_path, "subject": meta["subject"], "status": "GAGAL", "reason": "Tidak ada task/marker ditemukan"})
        loader._cleanup_tmp()
        continue
    
    # 7. Compute features (persis seperti web)
    feat_df = EEGFeatures.compute_first_occurrence_features(
        loader, df, ch_list, tasks_found, SELECTED_SUBBANDS, DEFAULT_FEATURES,
        include_frequency=True,
    )
    loader._cleanup_tmp()
    
    if feat_df.empty:
        # Diagnosa detail
        occurrences = loader.get_task_occurrences()
        detail_parts = []
        for task in tasks_found:
            task_occs = [o for o in occurrences if o["task"] == task]
            if not task_occs:
                detail_parts.append(f"  {task}: Tidak ada occurrence")
                continue
            for occ in task_occs:
                seg = loader.extract_occurrence_segment(df, task, occ["occurrence"])
                threshold = min(100, loader.sfreq * 0.5)
                detail_parts.append(f"  {task} occ{occ['occurrence']}: {len(seg)} sampel (min={int(threshold)})")
        
        reason = "Semua occurrence terlalu pendek / tidak bisa dihitung:\n" + "\n".join(detail_parts)
        results.append({"file": edf_path, "subject": meta["subject"], "status": "KOSONG", "reason": reason})
    else:
        tasks_extracted = feat_df["task"].unique().tolist()
        results.append({"file": edf_path, "subject": meta["subject"], "status": "SUKSES", "reason": f"Tasks: {tasks_extracted}, {len(feat_df)} baris fitur"})

# Tampilkan ringkasan
print("=" * 80)
print("RINGKASAN DIAGNOSTIK BATCH (Time1, Amplitude+ICA, Mu/Low_Beta/High_Beta)")
print("=" * 80)

sukses = [r for r in results if r["status"] == "SUKSES"]
gagal = [r for r in results if r["status"] != "SUKSES"]

print(f"\n✅ SUKSES: {len(sukses)} file")
print(f"❌ GAGAL/KOSONG: {len(gagal)} file\n")

if gagal:
    print("--- DETAIL FILE YANG GAGAL / KOSONG ---")
    for r in gagal:
        print(f"\n📁 {r['file']}")
        print(f"   Subject: {r['subject']}")
        print(f"   Status: {r['status']}")
        print(f"   Alasan: {r['reason']}")

print("\n--- DETAIL FILE SUKSES ---")
for r in sukses:
    print(f"  ✅ {r['subject']:12} | {r['file']}")
    print(f"     {r['reason']}")

Total file EDF ditemukan: 177

RINGKASAN DIAGNOSTIK BATCH (Time1, Amplitude+ICA, Mu/Low_Beta/High_Beta)

✅ SUKSES: 28 file
❌ GAGAL/KOSONG: 1 file

--- DETAIL FILE YANG GAGAL / KOSONG ---

📁 dummy/ALS03/time1/scenario3/EEG.edf
   Subject: ALS03
   Status: GAGAL
   Alasan: Tidak ada task/marker ditemukan

--- DETAIL FILE SUKSES ---
  ✅ ALS02        | dummy/ALS02/time1/scenario1/EEG.edf
     Tasks: [np.str_('Resting'), np.str_('Thinking'), np.str_('Typing')], 27 baris fitur
  ✅ ALS02        | dummy/ALS02/time1/scenario2/EEG.edf
     Tasks: [np.str_('Resting'), np.str_('Thinking'), np.str_('Typing')], 27 baris fitur
  ✅ ALS02        | dummy/ALS02/time1/scenario3/EEG.edf
     Tasks: [np.str_('Resting'), np.str_('Thinking'), np.str_('Typing')], 27 baris fitur
  ✅ ALS02        | dummy/ALS02/time1/scenario5/EEG.edf
     Tasks: [np.str_('Resting'), np.str_('Thinking'), np.str_('Typing')], 27 baris fitur
  ✅ ALS02        | dummy/ALS02/time1/scenario6/EEG.edf
     Tasks: [np.str_('Resting'), np.s

In [19]:
# Cek isi ZIP: list semua file EDF per subject di time1
import io

ZIP_PATH = r"C:\Users\Henry\Documents\KULIAH\riset\EEG\dummy.zip"

with open(ZIP_PATH, "rb") as f:
    zip_bytes = f.read()

edf_list = EEGLoader.list_edf_in_zip(io.BytesIO(zip_bytes))

# Group by subject
from collections import defaultdict
subject_files = defaultdict(list)

for edf_path in edf_list:
    meta = EEGLoader.detect_category(edf_path)
    if meta["time"] == "time1":
        subject_files[meta["subject"]].append(meta["scenario"])

print("=== FILE EDF YANG ADA DI ZIP (time1 saja) ===\n")
for subj in sorted(subject_files.keys()):
    scenarios = sorted(subject_files[subj])
    all_scenarios = [f"scenario{i}" for i in range(1, 10)]
    missing = [s for s in all_scenarios if s not in scenarios]
    print(f"{subj}:")
    print(f"  Ada     : {scenarios}")
    print(f"  Tidak ada: {missing if missing else '(lengkap semua)'}")
    print()

=== FILE EDF YANG ADA DI ZIP (time1 saja) ===

ALS02:
  Ada     : ['scenario1', 'scenario2', 'scenario3', 'scenario5', 'scenario6', 'scenario7', 'scenario9']
  Tidak ada: ['scenario4', 'scenario8']

ALS03:
  Ada     : ['scenario1', 'scenario2', 'scenario3', 'scenario5', 'scenario6', 'scenario8', 'scenario9']
  Tidak ada: ['scenario4', 'scenario7']

id2:
  Ada     : ['scenario1', 'scenario3', 'scenario4', 'scenario6', 'scenario8', 'scenario9']
  Tidak ada: ['scenario2', 'scenario5', 'scenario7']

id3:
  Ada     : ['scenario1', 'scenario2', 'scenario3', 'scenario4', 'scenario5', 'scenario6', 'scenario7', 'scenario8', 'scenario9']
  Tidak ada: (lengkap semua)

